# Генерация текста (GPT) - как нейросети учатся писать

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + трансформеры)
**Время прохождения:** ~120-150 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **чем** GPT отличается от BERT и почему это важно;
- разберёте **каузальное (маскированное) внимание** шаг за шагом;
- увидите **архитектуру GPT** слой за слоем;
- поработаете с **токенизацией BPE** на практике;
- загрузите **GPT-2** через HuggingFace и сгенерируете текст;
- сравните **стратегии генерации**: greedy, beam search, top-k, top-p;
- **обучите мини-GPT** с нуля на простом датасете;
- визуализируете **распределения вероятностей** при сэмплировании;
- поймёте эволюцию от **GPT-2 до GPT-4**.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | От BERT к GPT | Энкодер vs декодер, авторегрессионная генерация |
| 3 | Кказуальное (маскированное) внимание | Почему GPT не заглядывает вперёд |
| 4 | Архитектура GPT: декодер трансформера | Слой за слоем |
| 5 | Токенизация для GPT: BPE на практике | tiktoken, токенизатор GPT |
| 6 | Загрузка GPT-2 через HuggingFace | Шаг за шагом с комментариями |
| 7 | Стратегии генерации текста | greedy, beam search, top-k, top-p, temperature |
| 8 | Интерактивный генератор текста | Виджеты: temperature, top-k, top-p |
| 9 | Обучение мини-GPT с нуля | Маленькая модель на простом датасете |
| 10 | Управление генерацией: промпты | Как направлять GPT |
| 11 | Визуализация сэмплирования | Распределения вероятностей на каждом шаге |
| 12 | GPT-2 vs GPT-3 vs GPT-4 | Эволюция языковых моделей |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Создание и обучение моделей GPT |
| **transformers** | Предобученные модели (GPT-2) от HuggingFace |
| **tiktoken** | Быстрая токенизация BPE от OpenAI |
| **matplotlib** | Визуализация: графики, распределения, маски |
| **ipywidgets** | Интерактивные виджеты для генерации |

In [ ]:
# ============================================================
# ШАГ 1: Импортируем все нужные библиотеки
# ============================================================

import numpy as np                              # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                 # для построения графиков и визуализаций
import torch                                    # PyTorch - основной фреймворк
import torch.nn as nn                           # нейросетевые слои (Linear, LayerNorm...)
import torch.nn.functional as F                 # функции активации, softmax, и т.д.
import math                                     # математические функции (sqrt, pi)
from matplotlib.colors import LinearSegmentedColormap  # кастомные цветовые карты
from IPython.display import display, HTML       # красивый вывод в ноутбуке
import ipywidgets as widgets                    # интерактивные виджеты
from ipywidgets import interact, interactive, fixed  # декораторы для интерактива
import json                                     # для работы с JSON форматом

# Настройка matplotlib
plt.rcParams['figure.figsize'] = (10, 6)        # размер графиков по умолчанию
plt.rcParams['font.size'] = 12                  # размер шрифта
plt.rcParams['axes.grid'] = True                # показываем сетку

# Фиксируем random seed для воспроизводимости
torch.manual_seed(42)                           # seed для PyTorch
np.random.seed(42)                              # seed для NumPy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU или CPU
print(f"PyTorch: {torch.__version__}")
print(f"Устройство: {device}")
print("Все библиотеки импортированы!")

In [ ]:
# ============================================================
# ШАГ 1 (продолжение): Установка и импорт transformers и tiktoken
# ============================================================

# Устанавливаем библиотеки, если их нет (Colab обычно имеет их)
!pip install -q transformers tiktoken           # тихая установка transformers и tiktoken

from transformers import GPT2LMHeadModel, GPT2Tokenizer  # модель GPT-2 и её токенизатор
from transformers import pipeline               # удобный пайплайн для генерации текста
import tiktoken                                 # быстрая BPE-токенизация от OpenAI

print("transformers и tiktoken готовы!")

---
## Шаг 2. От BERT к GPT - энкодер vs декодер, авторегрессионная генерация

В предыдущих ноутбуках мы работали с **BERT** - энкодерной моделью. Теперь переходим к **GPT** - декодерной модели.

### Ключевое различие

| Свойство | BERT (энкодер) | GPT (декодер) |
|----------|---------------|---------------|
| Направление | Двунаправленный (видит весь текст) | Однонаправленный (видит только прошлое) |
| Задача | Маскированное предсказание (MLM) | Предсказание следующего токена |
| Маска внимания | Нет маски (все видят всех) | Кказуальная маска (только прошлые токены) |
| Применение | Классификация, NER, QA | Генерация текста, диалоги |
| Обучение | MLM + NSP | Next token prediction |

### Авторегрессионная генерация

GPT генерирует текст **по одному токену за раз**:
1. Подать промпт -> модель предсказывает следующий токен
2. Добавить предсказанный токен к промпту
3. Повторить

Это называется **авторегрессией** - каждый новый токен зависит от всех предыдущих.

In [ ]:
# ============================================================
# ШАГ 2: Демонстрация авторегрессионной генерации
# ============================================================

# Показываем концепцию авторегрессии на простом примере
# Модель генерирует по одному токену, каждый раз добавляя его к контексту

def demonstrate_autoregressive():
    # Демонстрация: как GPT генерирует текст шаг за шагом
    prompt = "Однажды в лесу"                          # начальный промпт
    generated_tokens = [" старик", " шёл", " по", " тропинке"]  # "сгенерированные" токены

    print("Авторегрессионная генерация текста:")
    print("=" * 50)
    print(f"Шаг 0 (промпт): '{prompt}'")               # начальный контекст

    # Показываем каждый шаг генерации
    current_text = prompt                                # текущий текст начинается с промпта
    for i, token in enumerate(generated_tokens, 1):     # перебираем "сгенерированные" токены
        current_text += token                            # добавляем новый токен к контексту
        print(f"Шаг {i}: контекст='{current_text[:-len(token)]}' -> новый токен='{token}'")
        print(f"        результат: '{current_text}'")    # показываем текущий результат

    print("=" * 50)
    print("Каждый новый токен зависит от ВСЕХ предыдущих!")
    print("Это и есть авторегрессия.")

demonstrate_autoregressive()                            # запускаем демонстрацию

---
## Шаг 3. Кказуальное (маскированное) self-attention - почему GPT не заглядывает вперёд

В BERT каждый токен может "видеть" все остальные токены. В GPT это **нельзя** допустить - иначе модель будет "подглядывать" в будущее.

### Решение: каузальная маска

Мы создаём **верхнетреугольную маску** из $-\infty$, которая после softmax превращается в 0.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right) V$$

| Позиция | Видит токены | Не видит |
|---------|-------------|----------|
| 1 | 1 | 2, 3, 4 |
| 2 | 1, 2 | 3, 4 |
| 3 | 1, 2, 3 | 4 |
| 4 | 1, 2, 3, 4 | - |

In [ ]:
# ============================================================
# ШАГ 3: Визуализация каузальной маски внимания
# ============================================================

# Создаём каузальную маску для последовательности длиной 8
seq_len = 8                                          # длина последовательности

# Создаём нижнетреугольную матрицу (1 = можно видеть, 0 = нельзя)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))  # нижний треугольник из единиц

print("Каузальная маска (1 = токен виден, 0 = токен скрыт):")
print(causal_mask.numpy())                            # выводим маску в виде numpy массива

# Визуализация каузальной маски
fig, axes = plt.subplots(1, 3, figsize=(18, 5))      # три графика в ряд

# График 1: Бинарная маска (что видно)
ax1 = axes[0]                                        # первая ось
im1 = ax1.imshow(causal_mask.numpy(), cmap='RdYlGn', vmin=0, vmax=1)  # тепловая карта маски
ax1.set_title('Что видит каждый токен\n(зелёный = видно, красный = скрыто)', fontsize=12)
ax1.set_xlabel('Ключ (позиция)', fontsize=11)        # подпись оси X
ax1.set_ylabel('Запрос (позиция)', fontsize=11)      # подпись оси Y

# Добавляем текстовые значения в ячейки
for i in range(seq_len):                              # перебираем строки
    for j in range(seq_len):                          # перебираем столбцы
        val = int(causal_mask[i, j].item())           # значение маски
        color = 'white' if val == 0 else 'black'      # цвет текста для читаемости
        ax1.text(j, i, str(val), ha='center', va='center', color=color, fontsize=10)

# График 2: Маска из -inf (что подаётся в softmax)
neg_inf_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)  # верхний треугольник = -inf
ax2 = axes[1]                                        # вторая ось
im2 = ax2.imshow(neg_inf_mask.numpy(), cmap='coolwarm', vmin=-5, vmax=0)  # тепловая карта -inf маски
ax2.set_title('Маска для softmax\n(-inf = заблокировано)', fontsize=12)
ax2.set_xlabel('Ключ (позиция)', fontsize=11)        # подпись оси X
ax2.set_ylabel('Запрос (позиция)', fontsize=11)      # подпись оси Y

# Добавляем текстовые значения
for i in range(seq_len):                              # перебираем строки
    for j in range(seq_len):                          # перебираем столбцы
        val = neg_inf_mask[i, j].item()               # значение маски
        text = '-inf' if val == float('-inf') else '0'  # текст для отображения
        color = 'white' if val == float('-inf') else 'black'  # цвет текста
        ax2.text(j, i, text, ha='center', va='center', color=color, fontsize=8)

# График 3: Результат после softmax
# Симулируем attention scores и применяем маску
scores = torch.randn(seq_len, seq_len)                # случайные скоры внимания
masked_scores = scores + neg_inf_mask                 # добавляем маску к скорам
attention_weights = F.softmax(masked_scores, dim=-1)  # применяем softmax по строкам

ax3 = axes[2]                                        # третья ось
im3 = ax3.imshow(attention_weights.numpy(), cmap='Blues', vmin=0, vmax=1)  # тепловая карта весов
ax3.set_title('Веса внимания после softmax\n(0 = невозможно)', fontsize=12)
ax3.set_xlabel('Ключ (позиция)', fontsize=11)        # подпись оси X
ax3.set_ylabel('Запрос (позиция)', fontsize=11)      # подпись оси Y

# Добавляем текстовые значения весов
for i in range(seq_len):                              # перебираем строки
    for j in range(seq_len):                          # перебираем столбцы
        val = attention_weights[i, j].item()          # значение веса
        if val > 0.01:                                # показываем только значимые веса
            ax3.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()                                    # автоматическая подстройка отступов
plt.savefig('causal_mask.png', dpi=150, bbox_inches='tight')  # сохраняем график
plt.show()                                            # отображаем график
print("Визуализация каузальной маски готова!")

---
## Шаг 4. Архитектура GPT: декодер трансформера - слой за слоем

GPT - это стек из **одинаковых блоков декодера**. Каждый блок содержит:

1. **Causal Self-Attention** - внимание с маской (не заглядывает вперёд)
2. **Layer Normalization** - нормализация перед и после внимания
3. **Feed-Forward Network** - двухслойная сеть с GELU активацией
4. **Residual connections** - остаточные связи вокруг каждого под-слоя

### Архитектура GPT-2 (Small)

```
Входные токены -> Token Embedding + Position Embedding
    |
    v
[Block 1: CausalAttention -> Add&Norm -> FFN -> Add&Norm]
    |
    v
[Block 2: CausalAttention -> Add&Norm -> FFN -> Add&Norm]
    |
    v
... (12 блоков для GPT-2 Small)
    |
    v
Layer Norm -> Linear (vocab_size) -> softmax
    |
    v
Распределение вероятностей следующего токена
```

### Параметры GPT-2 Small
- 12 слоёв (blocks)
- 768 размер скрытого слоя
- 12 голов внимания
- 50257 размер словаря
- 1024 максимальная длина контекста
- ~124M параметров

In [ ]:
# ============================================================
# ШАГ 4: Реализация архитектуры GPT с нуля
# ============================================================

class CausalSelfAttention(nn.Module):
    # Кказуальное self-attention: токены видят только прошлые позиции
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd должно делиться на n_head"
        self.n_head = n_head                          # количество голов внимания
        self.n_embd = n_embd                          # размер эмбеддинга
        self.head_dim = n_embd // n_head              # размер каждой головы

        # Одна проекция для Q, K, V (объединённая для эффективности)
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)  # проекция Q, K, V одновременно
        self.c_proj = nn.Linear(n_embd, n_embd)       # выходная проекция
        self.attn_dropout = nn.Dropout(dropout)        # dropout для весов внимания
        self.resid_dropout = nn.Dropout(dropout)       # dropout для остаточной связи

        # Регистрируем каузальную маску как буфер (не обучается)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                            .view(1, 1, block_size, block_size))  # нижнетреугольная маска

    def forward(self, x):
        B, T, C = x.size()                            # batch, seq_len, n_embd

        # Вычисляем Q, K, V за одну проекцию
        qkv = self.c_attn(x)                          # (B, T, 3*C) - объединённые Q,K,V
        q, k, v = qkv.split(self.n_embd, dim=2)      # разделяем на Q, K, V каждый (B, T, C)

        # Переформатируем для multi-head: (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)

        # Вычисляем attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))  # (B, nh, T, T)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))  # маска: -inf где нельзя
        att = F.softmax(att, dim=-1)                  # softmax по последней размерности
        att = self.attn_dropout(att)                  # применяем dropout к весам

        # Умножаем веса на значения
        y = att @ v                                   # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # собираем все головы обратно

        # Выходная проекция + dropout
        y = self.resid_dropout(self.c_proj(y))        # (B, T, C)
        return y                                      # возвращаем результат


class MLP(nn.Module):
    # Feed-forward сеть внутри блока GPT
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)    # расширение: 768 -> 3072
        self.c_proj = nn.Linear(4 * n_embd, n_embd)   # сужение: 3072 -> 768
        self.gelu = nn.GELU()                         # активация GELU (как в GPT-2)
        self.dropout = nn.Dropout(dropout)             # dropout для регуляризации

    def forward(self, x):
        x = self.c_fc(x)                              # расширяем размерность
        x = self.gelu(x)                              # применяем GELU активацию
        x = self.c_proj(x)                            # сужаем обратно
        x = self.dropout(x)                           # применяем dropout
        return x                                      # возвращаем результат


class Block(nn.Module):
    # Один блок декодера GPT: Attention + MLP с остаточными связями
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)              # LayerNorm перед attention
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)  # attention
        self.ln_2 = nn.LayerNorm(n_embd)              # LayerNorm перед MLP
        self.mlp = MLP(n_embd, dropout)               # feed-forward сеть

    def forward(self, x):
        # Остаточная связь: x + sublayer(norm(x)) - Pre-LN вариант
        x = x + self.attn(self.ln_1(x))              # attention с остаточной связью
        x = x + self.mlp(self.ln_2(x))               # MLP с остаточной связью
        return x                                      # возвращаем результат


class GPT(nn.Module):
    # Полная модель GPT
    def __init__(self, vocab_size, n_embd=768, n_head=12, n_layer=12,
                 block_size=1024, dropout=0.1):
        super().__init__()
        self.block_size = block_size                   # максимальная длина контекста

        # Токенные и позиционные эмбеддинги
        self.wte = nn.Embedding(vocab_size, n_embd)   # токенные эмбеддинги
        self.wpe = nn.Embedding(block_size, n_embd)   # позиционные эмбеддинги
        self.drop = nn.Dropout(dropout)                # dropout после эмбеддингов

        # Стек блоков декодера
        self.blocks = nn.ModuleList([                  # список блоков
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        # Финальная нормализация и выходная проекция
        self.ln_f = nn.LayerNorm(n_embd)              # финальный LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)  # проекция в словарь

        # Инициализация весов
        self.apply(self._init_weights)                 # применяем инициализацию ко всем слоям

    def _init_weights(self, module):
        # Инициализация весов по GPT-2 paper
        if isinstance(module, nn.Linear):              # если слой линейный
            nn.init.normal_(module.weight, mean=0.0, std=0.02)  # нормальная инициализация
            if module.bias is not None:                # если есть смещение
                nn.init.zeros_(module.bias)            # инициализируем нулями
        elif isinstance(module, nn.Embedding):         # если слой эмбеддинга
            nn.init.normal_(module.weight, mean=0.0, std=0.02)  # нормальная инициализация

    def forward(self, idx, targets=None):
        B, T = idx.size()                              # batch, seq_len

        # Токенные + позиционные эмбеддинги
        tok_emb = self.wte(idx)                        # (B, T, n_embd) - токенные эмбеддинги
        pos = torch.arange(0, T, device=idx.device)   # позиции от 0 до T-1
        pos_emb = self.wpe(pos)                        # (T, n_embd) - позиционные эмбеддинги
        x = self.drop(tok_emb + pos_emb)               # суммируем и применяем dropout

        # Пропускаем через все блоки
        for block in self.blocks:                      # перебираем блоки
            x = block(x)                               # применяем блок

        x = self.ln_f(x)                              # финальная нормализация
        logits = self.lm_head(x)                       # (B, T, vocab_size) - логиты

        # Вычисляем loss если есть целевые токены
        loss = None
        if targets is not None:                        # если переданы целевые токены
            loss = F.cross_entropy(                    # кросс-энтропия
                logits.view(-1, logits.size(-1)),      # выравниваем логиты
                targets.view(-1)                       # выравниваем целевые токены
            )
        return logits, loss                            # возвращаем логиты и loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        # Авторегрессионная генерация текста
        for _ in range(max_new_tokens):                # генерируем заданное число токенов
            idx_cond = idx[:, -self.block_size:]       # обрезаем контекст до block_size
            logits, _ = self(idx_cond)                 # получаем логиты от модели
            logits = logits[:, -1, :]                  # берём логиты последнего токена
            logits = logits / temperature              # применяем temperature
            if top_k is not None:                      # если указан top-k
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))  # top-k значений
                logits[logits < v[:, [-1]]] = float('-inf')  # отсекаем всё за top-k
            probs = F.softmax(logits, dim=-1)          # преобразуем в вероятности
            idx_next = torch.multinomial(probs, num_samples=1)  # сэмплируем следующий токен
            idx = torch.cat((idx, idx_next), dim=1)    # добавляем к последовательности
        return idx                                    # возвращаем результат


print("Архитектура GPT реализована с нуля!")
print("Классы: CausalSelfAttention, MLP, Block, GPT")

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Визуализация архитектуры GPT
# ============================================================

# Создаём схему архитектуры GPT с помощью matplotlib
fig, ax = plt.subplots(1, 1, figsize=(14, 10))       # создаём фигуру
ax.set_xlim(0, 10)                                    # пределы по X
ax.set_ylim(0, 14)                                    # пределы по Y
ax.set_aspect('equal')                                # одинаковый масштаб
ax.axis('off')                                        # убираем оси

# Определяем цвета для компонентов
colors = {
    'embedding': '#4ECDC4',   # бирюзовый для эмбеддингов
    'attention': '#FF6B6B',   # красный для attention
    'ffn': '#45B7D1',         # голубой для FFN
    'norm': '#96CEB4',        # зелёный для LayerNorm
    'output': '#FFEAA7',      # жёлтый для выхода
}

# Рисуем входные эмбеддинги
rect = plt.Rectangle((2, 12.5), 6, 0.7, facecolor=colors['embedding'], edgecolor='black', lw=2)
ax.add_patch(rect)                                    # добавляем прямоугольник
ax.text(5, 12.85, 'Token Embedding + Position Embedding', ha='center', va='center', fontsize=10, fontweight='bold')

# Рисуем блоки декодера
block_labels = ['Block N', 'Block 2', 'Block 1']
y_positions = [9.0, 6.5, 4.0]                        # Y координаты блоков

for i, (label, y) in enumerate(zip(block_labels, y_positions)):
    # Рамка блока
    rect = plt.Rectangle((1.5, y), 7, 2, facecolor='#F8F9FA', edgecolor='black', lw=2)
    ax.add_patch(rect)                                # добавляем прямоугольник блока
    ax.text(5, y + 1.7, label, ha='center', va='center', fontsize=11, fontweight='bold')

    # LayerNorm + Attention
    rect1 = plt.Rectangle((2, y + 0.7), 2.5, 0.7, facecolor=colors['norm'], edgecolor='black', lw=1)
    ax.add_patch(rect1)                               # LayerNorm
    ax.text(3.25, y + 1.05, 'LayerNorm', ha='center', va='center', fontsize=8)

    rect2 = plt.Rectangle((5, y + 0.7), 3, 0.7, facecolor=colors['attention'], edgecolor='black', lw=1)
    ax.add_patch(rect2)                               # Attention
    ax.text(6.5, y + 1.05, 'Causal Attn', ha='center', va='center', fontsize=8, color='white')

    # LayerNorm + FFN
    rect3 = plt.Rectangle((2, y), 2.5, 0.6, facecolor=colors['norm'], edgecolor='black', lw=1)
    ax.add_patch(rect3)                               # LayerNorm
    ax.text(3.25, y + 0.3, 'LayerNorm', ha='center', va='center', fontsize=8)

    rect4 = plt.Rectangle((5, y), 3, 0.6, facecolor=colors['ffn'], edgecolor='black', lw=1)
    ax.add_patch(rect4)                               # FFN
    ax.text(6.5, y + 0.3, 'FFN (MLP)', ha='center', va='center', fontsize=8, color='white')

    # Стрелка (+ residual)
    ax.annotate('', xy=(5, y + 0.7), xytext=(5, y + 2),
               arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))  # стрелка вниз

# Стрелки между блоками
ax.annotate('', xy=(5, 11.0), xytext=(5, 12.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стрелка вниз
ax.annotate('', xy=(5, 8.5), xytext=(5, 9.0),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стрелка вниз
ax.annotate('', xy=(5, 6.0), xytext=(5, 6.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стрелка вниз
ax.text(8.5, 9.0, '... x 12', ha='center', fontsize=14, fontweight='bold', color='gray')

# Выходные слои
rect5 = plt.Rectangle((2.5, 2.5), 5, 0.7, facecolor=colors['norm'], edgecolor='black', lw=2)
ax.add_patch(rect5)                                   # финальный LayerNorm
ax.text(5, 2.85, 'Final LayerNorm', ha='center', va='center', fontsize=10, fontweight='bold')

rect6 = plt.Rectangle((2.5, 1.5), 5, 0.7, facecolor=colors['output'], edgecolor='black', lw=2)
ax.add_patch(rect6)                                   # линейный выход
ax.text(5, 1.85, 'Linear (vocab_size) -> Softmax', ha='center', va='center', fontsize=10, fontweight='bold')

# Стрелки к выходу
ax.annotate('', xy=(5, 3.2), xytext=(5, 4.0),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стрелка вниз
ax.annotate('', xy=(5, 2.2), xytext=(5, 2.5),
           arrowprops=dict(arrowstyle='->', color='black', lw=2))  # стрелка вниз

# Подпись
ax.text(5, 0.5, 'P(next token | context)', ha='center', fontsize=12,
       fontstyle='italic', fontweight='bold', color='#2C3E50')

ax.set_title('Архитектура GPT: Декодер Трансформера', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('gpt_architecture.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Схема архитектуры GPT готова!")

---
## Шаг 5. Токенизация для GPT: BPE на практике

GPT использует **Byte Pair Encoding (BPE)** - алгоритм токенизации, который:
1. Начинает с отдельных байтов как базовых токенов
2. Итеративно объединяет самые частые пары токенов
3. Создаёт словарь подслов (subwords)

### Почему BPE?
- Не требует пробелов (работает с любым языком)
- Баланс между размером словаря и длиной последовательности
- Обрабатывает неизвестные слова через подслова

### GPT-2 использует BPE с словарём 50257 токенов

In [ ]:
# ============================================================
# ШАГ 5: BPE токенизация - tiktoken и GPT-2 tokenizer
# ============================================================

# Пример 1: Используем tiktoken (быстрая BPE токенизация от OpenAI)
enc = tiktoken.get_encoding("gpt2")                   # загружаем кодировку GPT-2

# Токенизируем пример текста
text = "Hello, world! This is GPT in action."         # пример текста
tokens = enc.encode(text)                             # кодируем текст в токены (список ID)

print(f"Текст: '{text}'")                             # выводим исходный текст
print(f"Токены (ID): {tokens}")                       # выводим список ID токенов
print(f"Количество токенов: {len(tokens)}")            # выводим количество токенов

# Декодируем обратно
decoded = enc.decode(tokens)                          # декодируем токены обратно в текст
print(f"Декодировано: '{decoded}'")                   # выводим декодированный текст

# Посмотрим, что представляет каждый токен
print("\nРазбор токенов:")
for token_id in tokens:                               # перебираем все ID токенов
    token_bytes = enc.decode_single_byte_string(token_id) if False else enc.decode([token_id])  # декодируем один токен
    print(f"  ID={token_id:5d} -> '{token_bytes}'")   # выводим ID и текст токена

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Визуализация BPE токенизации
# ============================================================

# Сравниваем токенизацию разных текстов
texts = [
    "Hello world",                                    # простой английский
    "Привет мир",                                     # русский
    "GPT-2 is great!",                                # с пунктуацией
    "supercalifragilisticexpialidocious",              # длинное слово
    "https://example.com/path?id=123",                # URL
]

fig, axes = plt.subplots(len(texts), 1, figsize=(14, 2.5 * len(texts)))  # создаем фигуру
if len(texts) == 1:
    axes = [axes]                                     # обеспечиваем список для одного текста

colors_list = plt.cm.Set3(np.linspace(0, 1, 50))     # цвета для токенов

for idx, text in enumerate(texts):                    # перебираем тексты
    ax = axes[idx]                                    # текущая ось
    tokens = enc.encode(text)                         # токенизируем текст
    token_strs = [enc.decode([t]) for t in tokens]    # декодируем каждый токен отдельно

    # Рисуем токены как цветные блоки
    x_pos = 0                                         # начальная позиция по X
    for j, (tid, tstr) in enumerate(zip(tokens, token_strs)):
        width = max(len(tstr) * 0.15 + 0.3, 0.5)     # ширина блока зависит от длины токена
        color = colors_list[j % len(colors_list)]     # цвет токена
        rect = plt.Rectangle((x_pos, 0.2), width, 0.6,  # рисуем прямоугольник
                            facecolor=color, edgecolor='black', lw=1.5)
        ax.add_patch(rect)                            # добавляем прямоугольник
        # Подпись: текст токена
        display_str = tstr.replace(' ', '_')          # заменяем пробелы для наглядности
        ax.text(x_pos + width/2, 0.5, display_str,   # текст внутри блока
               ha='center', va='center', fontsize=9, fontweight='bold')
        # ID токена сверху
        ax.text(x_pos + width/2, 0.85, str(tid),     # ID токена над блоком
               ha='center', va='center', fontsize=7, color='gray')
        x_pos += width + 0.05                         # сдвигаем позицию

    ax.set_xlim(-0.2, x_pos + 0.2)                   # пределы по X
    ax.set_ylim(0, 1.1)                               # пределы по Y
    ax.set_title(f'Текст: "{text}" -> {len(tokens)} токенов', fontsize=10)  # заголовок
    ax.axis('off')                                    # убираем оси

plt.suptitle('BPE токенизация: текст -> токены GPT-2', fontsize=14, fontweight='bold')
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('bpe_tokenization.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация BPE готова!")

---
## Шаг 6. Загрузка GPT-2 с HuggingFace - шаг за шагом

HuggingFace Transformers предоставляет предобученные модели GPT-2. Загрузим модель и токенизатор.

### Что загружаем:
- **GPT2LMHeadModel** - модель GPT-2 с головой для генерации
- **GPT2Tokenizer** - токенизатор BPE для GPT-2

### Варианты GPT-2:
| Модель | Параметры | n_layer | n_head | n_embd |
|--------|-----------|---------|--------|--------|
| gpt2 | 124M | 12 | 12 | 768 |
| gpt2-medium | 355M | 24 | 16 | 1024 |
| gpt2-large | 774M | 36 | 20 | 1280 |
| gpt2-xl | 1.5B | 48 | 25 | 1600 |

In [ ]:
# ============================================================
# ШАГ 6: Загрузка GPT-2 через HuggingFace
# ============================================================

# Загружаем модель и токенизатор GPT-2 (Small, 124M параметров)
model_name = "gpt2"                                   # имя модели (маленькая версия)
print(f"Загружаем модель: {model_name}...")

tokenizer = GPT2Tokenizer.from_pretrained(model_name) # загружаем токенизатор GPT-2
model = GPT2LMHeadModel.from_pretrained(model_name)   # загружаем модель GPT-2
model.eval()                                          # переводим в режим оценки (без обучения)

# Перемещаем модель на GPU если доступно
model.to(device)                                      # перемещаем на GPU/CPU
print(f"Модель загружена на {device}!")

# Смотрим размер модели
total_params = sum(p.numel() for p in model.parameters())  # общее число параметров
print(f"Количество параметров: {total_params:,}")     # выводим количество параметров

# Смотрим конфигурацию модели
print(f"\nКонфигурация GPT-2:")
print(f"  Слои (n_layer): {model.config.n_layer}")    # количество слоёв
print(f"  Головы (n_head): {model.config.n_head}")    # количество голов внимания
print(f"  Эмбеддинг (n_embd): {model.config.n_embd}") # размер эмбеддинга
print(f"  Словарь (vocab_size): {model.config.vocab_size}")  # размер словаря
print(f"  Контекст (n_positions): {model.config.n_positions}")  # максимальная длина контекста

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Первая генерация текста с GPT-2
# ============================================================

# Задаём начальный промпт для генерации
prompt = "Once upon a time"                            # начальный текст
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем и переносим

print(f"Промпт: '{prompt}'")                          # выводим промпт
print(f"Токены: {input_ids[0].tolist()}")             # выводим ID токенов

# Генерируем текст с настройками по умолчанию
output_ids = model.generate(                           # генерируем текст
    input_ids,                                         # входные токены
    max_length=100,                                    # максимальная длина выхода
    do_sample=True,                                    # включаем сэмплирование (не greedy)
    temperature=0.8,                                   # temperature < 1 = более фокусированный
    top_k=50,                                          # top-k сэмплирование
    pad_token_id=tokenizer.eos_token_id                # ID токена конца последовательности
)

# Декодируем и выводим сгенерированный текст
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)  # декодируем
print(f"\nСгенерированный текст:")
print("-" * 50)
print(generated_text)                                 # выводим результат
print("-" * 50)

---
## Шаг 7. Стратегии генерации текста

GPT выдаёт **распределение вероятностей** над словарём. Как выбрать следующий токен?

### Основные стратегии:

1. **Greedy** - выбираем самый вероятный токен. Просто, но скучно и повторяется.
2. **Beam Search** - держим несколько лучших гипотез. Лучше, но всё равно скучно.
3. **Top-k** - выбираем из k самых вероятных токенов. Разнообразнее.
4. **Top-p (Nucleus)** - выбираем из минимального набора токенов с суммарной вероятностью >= p. Адаптивно.
5. **Temperature** - контролируем "остроту" распределения:
   - T < 1: более определённые, предсказуемые выборы
   - T = 1: исходное распределение
   - T > 1: более случайные, креативные выборы

In [ ]:
# ============================================================
# ШАГ 7: Сравнение стратегий генерации
# ============================================================

# Функция для генерации текста с разными стратегиями
def generate_with_strategy(prompt, strategy="greedy", max_length=80,
                           temperature=1.0, top_k=50, top_p=0.9,
                           num_beams=5, num_return_sequences=1):
    # Кодируем промпт
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем

    # Настраиваем параметры генерации в зависимости от стратегии
    if strategy == "greedy":                           # жадная стратегия
        output = model.generate(                      # генерируем
            input_ids,
            max_length=max_length,                    # максимальная длина
            do_sample=False,                          # без сэмплирования
            pad_token_id=tokenizer.eos_token_id       # токен паддинга
        )
    elif strategy == "beam":                          # beam search
        output = model.generate(                      # генерируем
            input_ids,
            max_length=max_length,                    # максимальная длина
            num_beams=num_beams,                      # количество лучей
            num_return_sequences=num_return_sequences,  # количество возвращаемых последовательностей
            pad_token_id=tokenizer.eos_token_id       # токен паддинга
        )
    elif strategy == "top_k":                        # top-k сэмплирование
        output = model.generate(                      # генерируем
            input_ids,
            max_length=max_length,                    # максимальная длина
            do_sample=True,                           # со сэмплированием
            top_k=top_k,                              # top-k параметр
            temperature=temperature,                  # temperature
            pad_token_id=tokenizer.eos_token_id       # токен паддинга
        )
    elif strategy == "top_p":                        # nucleus сэмплирование
        output = model.generate(                      # генерируем
            input_ids,
            max_length=max_length,                    # максимальная длина
            do_sample=True,                           # со сэмплированием
            top_p=top_p,                              # top-p параметр
            temperature=temperature,                  # temperature
            pad_token_id=tokenizer.eos_token_id       # токен паддинга
        )

    # Декодируем и возвращаем результаты
    results = []
    for i in range(output.shape[0]):                  # перебираем все сгенерированные последовательности
        text = tokenizer.decode(output[i], skip_special_tokens=True)  # декодируем
        results.append(text)                          # добавляем в список
    return results                                    # возвращаем все результаты


# Сравниваем стратегии на одном промпте
prompt = "The future of artificial intelligence"       # один и тот же промпт
strategies = {                                        # словарь стратегий
    "Greedy": {"strategy": "greedy"},
    "Beam Search (5)": {"strategy": "beam", "num_beams": 5},
    "Top-k (k=50)": {"strategy": "top_k", "top_k": 50, "temperature": 0.8},
    "Top-p (p=0.9)": {"strategy": "top_p", "top_p": 0.9, "temperature": 0.8},
}

print(f"Промпт: '{prompt}'\n")
print("=" * 70)

for name, params in strategies.items():               # перебираем стратегии
    result = generate_with_strategy(prompt, **params)  # генерируем текст
    print(f"\n--- {name} ---")
    print(result[0][:200] + "..." if len(result[0]) > 200 else result[0])  # выводим первые 200 символов
    print()

print("=" * 70)
print("Сравнение стратегий завершено!")

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Визуализация стратегий генерации
# ============================================================

# Получаем логиты для промпта и визуализируем распределения
prompt = "The future of"                               # промпт
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем

with torch.no_grad():                                  # без вычисления градиентов
    outputs = model(input_ids)                         # получаем выход модели
    logits = outputs.logits[0, -1, :].cpu()            # логиты последнего токена

# Визуализируем эффект temperature и top-k/top-p на распределение
fig, axes = plt.subplots(2, 3, figsize=(18, 10))      # 2 строки, 3 столбца

# Топ-20 токенов по вероятности
probs_base = F.softmax(logits, dim=-1)                # исходное распределение
top_vals, top_indices = torch.topk(probs_base, 20)    # топ-20 токенов
token_labels = [tokenizer.decode([idx]) for idx in top_indices.tolist()]  # тексты токенов

# График 1: Исходное распределение (temperature=1.0)
ax = axes[0, 0]                                       # первая ячейка
ax.barh(range(20), top_vals.numpy(), color='#3498DB')  # горизонтальная столбчатая
ax.set_yticks(range(20))                              # деления по Y
ax.set_yticklabels(token_labels, fontsize=8)          # подписи токенов
ax.invert_yaxis()                                     # переворачиваем ось
ax.set_title('Temperature = 1.0\n(исходное распределение)', fontsize=11)
ax.set_xlabel('Вероятность')

# График 2: Temperature = 0.5
ax = axes[0, 1]                                       # вторая ячейка
probs_t05 = F.softmax(logits / 0.5, dim=-1)          # temperature = 0.5 (более острое)
top_vals_05 = probs_t05[top_indices]                  # вероятности тех же топ-20 токенов
ax.barh(range(20), top_vals_05.numpy(), color='#E74C3C')  # столбчатая диаграмма
ax.set_yticks(range(20))                              # деления по Y
ax.set_yticklabels(token_labels, fontsize=8)          # подписи токенов
ax.invert_yaxis()                                     # переворачиваем ось
ax.set_title('Temperature = 0.5\n(более определённое)', fontsize=11)
ax.set_xlabel('Вероятность')

# График 3: Temperature = 2.0
ax = axes[0, 2]                                       # третья ячейка
probs_t20 = F.softmax(logits / 2.0, dim=-1)          # temperature = 2.0 (более плоское)
top_vals_20 = probs_t20[top_indices]                  # вероятности тех же топ-20 токенов
ax.barh(range(20), top_vals_20.numpy(), color='#2ECC71')  # столбчатая диаграмма
ax.set_yticks(range(20))                              # деления по Y
ax.set_yticklabels(token_labels, fontsize=8)          # подписи токенов
ax.invert_yaxis()                                     # переворачиваем ось
ax.set_title('Temperature = 2.0\n(более случайное)', fontsize=11)
ax.set_xlabel('Вероятность')

# Графики для top-k и top-p
# График 4: Top-k = 5
ax = axes[1, 0]                                       # четвёртая ячейка
probs_topk = probs_base.clone()                       # копируем распределение
sorted_probs, sorted_idx = torch.sort(probs_topk, descending=True)  # сортируем
probs_topk[sorted_idx[5:]] = 0                        # обнуляем всё кроме топ-5
probs_topk = probs_topk / probs_topk.sum()            # перенормируем
top_vals_k5 = probs_topk[top_indices]                 # вероятности топ-20 после фильтрации
ax.barh(range(20), top_vals_k5.numpy(), color='#9B59B6')  # столбчатая
ax.set_yticks(range(20))                              # деления по Y
ax.set_yticklabels(token_labels, fontsize=8)          # подписи токенов
ax.invert_yaxis()                                     # переворачиваем ось
ax.set_title('Top-k = 5\n(только 5 лучших токенов)', fontsize=11)
ax.set_xlabel('Вероятность')

# График 5: Top-p = 0.9
ax = axes[1, 1]                                       # пятая ячейка
sorted_probs_p, sorted_idx_p = torch.sort(probs_base, descending=True)  # сортируем
cumsum_p = torch.cumsum(sorted_probs_p, dim=-1)       # кумулятивная сумма
cutoff = cumsum_p - sorted_probs_p > 0.9              # находим точку отсечения
sorted_probs_p[cutoff] = 0                            # обнуляем токены за порогом
probs_topp = torch.zeros_like(probs_base)             # создаём пустое распределение
probs_topp[sorted_idx_p] = sorted_probs_p             # восстанавливаем порядок
probs_topp = probs_topp / probs_topp.sum()            # перенормируем
top_vals_p09 = probs_topp[top_indices]                # вероятности топ-20 после фильтрации
ax.barh(range(20), top_vals_p09.numpy(), color='#F39C12')  # столбчатая
ax.set_yticks(range(20))                              # деления по Y
ax.set_yticklabels(token_labels, fontsize=8)          # подписи токенов
ax.invert_yaxis()                                     # переворачиваем ось
ax.set_title('Top-p = 0.9\n(накопленная вероятность 90%)', fontsize=11)
ax.set_xlabel('Вероятность')

# График 6: Сравнение всех стратегий (кумулятивная вероятность)
ax = axes[1, 2]                                       # шестая ячейка
sorted_base, _ = torch.sort(probs_base, descending=True)  # сортируем базовые вероятности
cumsum_base = torch.cumsum(sorted_base, dim=-1)       # кумулятивная сумма
sorted_t05, _ = torch.sort(probs_t05, descending=True)  # сортируем t=0.5
cumsum_t05 = torch.cumsum(sorted_t05, dim=-1)        # кумулятивная сумма
sorted_t20, _ = torch.sort(probs_t20, descending=True)  # сортируем t=2.0
cumsum_t20 = torch.cumsum(sorted_t20, dim=-1)        # кумулятивная сумма

n_show = 100                                          # показываем первые 100 токенов
ax.plot(range(n_show), cumsum_base[:n_show].numpy(), 'b-', label='T=1.0', lw=2)  # T=1.0
ax.plot(range(n_show), cumsum_t05[:n_show].numpy(), 'r-', label='T=0.5', lw=2)   # T=0.5
ax.plot(range(n_show), cumsum_t20[:n_show].numpy(), 'g-', label='T=2.0', lw=2)   # T=2.0
ax.axhline(y=0.9, color='gray', linestyle='--', label='p=0.9 порог')  # порог top-p
ax.set_xlabel('Количество токенов (отсортированных по вероятности)')
ax.set_ylabel('Кумулятивная вероятность')
ax.set_title('Кумулятивная вероятность\nпо топ-токенам', fontsize=11)
ax.legend(fontsize=9)                                 # легенда

plt.suptitle(f'Стратегии генерации для промпта: "{prompt}"', fontsize=14, fontweight='bold')
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('generation_strategies.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация стратегий генерации готова!")

---
## Шаг 8. Интерактивный генератор текста

Теперь создадим интерактивный генератор с виджетами для настройки:
- **Temperature** - от 0.1 (почти детерминировано) до 2.0 (очень случайно)
- **Top-k** - количество лучших токенов для выбора (1-100)
- **Top-p** - порог накопленной вероятности (0.1-1.0)
- **Max length** - максимальная длина генерации

In [ ]:
# ============================================================
# ШАГ 8: Интерактивный генератор текста с виджетами
# ============================================================

def interactive_generate(prompt, temperature, top_k, top_p, max_length):
    # Функция для интерактивной генерации текста
    if not prompt.strip():                            # если промпт пустой
        print("Введите промпт!")
        return                                       # выходим

    # Кодируем промпт
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем

    # Генерируем текст с заданными параметрами
    output_ids = model.generate(                      # генерируем
        input_ids,
        max_length=max_length,                        # максимальная длина
        do_sample=True,                               # включаем сэмплирование
        temperature=temperature,                      # temperature
        top_k=top_k if top_k > 0 else None,           # top-k (отключаем если 0)
        top_p=top_p,                                  # top-p
        pad_token_id=tokenizer.eos_token_id           # токен паддинга
    )

    # Декодируем и выводим
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)  # декодируем
    print(generated)                                  # выводим результат

# Создаём интерактивные виджеты
prompt_widget = widgets.Text(                         # текстовое поле для промпта
    value='In the year 2050, humanity',               # значение по умолчанию
    description='Промпт:',                            # подпись
    layout=widgets.Layout(width='80%')                # ширина
)

temperature_widget = widgets.FloatSlider(             # слайдер temperature
    value=0.8,                                        # значение по умолчанию
    min=0.1,                                          # минимум
    max=2.0,                                          # максимум
    step=0.1,                                         # шаг
    description='Temperature:',                       # подпись
    continuous_update=False                            # обновлять только при отпускании
)

top_k_widget = widgets.IntSlider(                     # слайдер top-k
    value=50,                                         # значение по умолчанию
    min=1,                                            # минимум
    max=100,                                          # максимум
    step=1,                                           # шаг
    description='Top-k:',                             # подпись
    continuous_update=False                            # обновлять только при отпускании
)

top_p_widget = widgets.FloatSlider(                   # слайдер top-p
    value=0.9,                                        # значение по умолчанию
    min=0.1,                                          # минимум
    max=1.0,                                          # максимум
    step=0.05,                                        # шаг
    description='Top-p:',                             # подпись
    continuous_update=False                            # обновлять только при отпускании
)

max_length_widget = widgets.IntSlider(                # слайдер максимальной длины
    value=100,                                        # значение по умолчанию
    min=20,                                           # минимум
    max=300,                                          # максимум
    step=10,                                          # шаг
    description='Max length:',                        # подпись
    continuous_update=False                            # обновлять только при отпускании
)

# Создаём интерактивный интерфейс
interactive_output = interactive(                     # интерактивный виджет
    interactive_generate,                             # функция генерации
    prompt=prompt_widget,                             # виджет промпта
    temperature=temperature_widget,                   # виджет temperature
    top_k=top_k_widget,                               # виджет top-k
    top_p=top_p_widget,                               # виджет top-p
    max_length=max_length_widget                      # виджет макс. длины
)

display(interactive_output)                           # отображаем интерфейс
print("Интерактивный генератор готов! Настройте параметры и измените промпт.")

---
## Шаг 9. Обучение мини-GPT с нуля

Теперь самое интересное - обучим маленькую GPT модель с нуля!

### Наша мини-GPT:
- 4 слоя (вместо 12 у GPT-2 Small)
- 4 головы внимания
- 128 размер эмбеддинга
- Контекст 256 токенов
- Обучаем на простых текстовых данных

Мы увидим, как модель постепенно учится предсказывать следующий символ.

In [ ]:
# ============================================================
# ШАГ 9: Обучение мини-GPT с нуля - подготовка данных
# ============================================================

# Создаём простой обучающий датасет из повторяющегося текста
# Модель должна научиться предсказывать следующий символ

# Простой текст для обучения (повторяющийся, чтобы модель быстрее выучила паттерн)
raw_text = "The quick brown fox jumps over the lazy dog. " * 50  # повторяем текст 50 раз
raw_text += "In a hole in the ground there lived a hobbit. " * 30  # добавляем другой текст
raw_text += "To be or not to be that is the question. " * 30     # и ещё один паттерн

print(f"Длина обучающего текста: {len(raw_text)} символов")  # выводим длину

# Создаём посимвольную токенизацию (самая простая)
chars = sorted(list(set(raw_text)))                   # уникальные символы в тексте
vocab_size_mini = len(chars)                          # размер словаря
print(f"Размер словаря (символов): {vocab_size_mini}")  # выводим размер словаря

# Создаём маппинги символ <-> число
char_to_idx = {ch: i for i, ch in enumerate(chars)}  # символ -> индекс
idx_to_char = {i: ch for i, ch in enumerate(chars)}  # индекс -> символ

# Кодируем весь текст в числа
data = torch.tensor([char_to_idx[ch] for ch in raw_text], dtype=torch.long)  # кодируем
print(f"Закодированных токенов: {data.shape[0]}")     # выводим количество токенов

# Разбиваем на обучающую и валидационную выборки
n_split = int(0.9 * len(data))                        # 90% для обучения
train_data = data[:n_split]                           # обучающая выборка
val_data = data[n_split:]                             # валидационная выборка
print(f"Обучающих токенов: {len(train_data):,}")      # выводим размер обучающей выборки
print(f"Валидационных токенов: {len(val_data):,}")    # выводим размер валидационной выборки

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Функция для получения батчей
# ============================================================

# Параметры обучения
block_size = 128                                      # длина контекста (сколько символов видит модель)
batch_size = 32                                       # размер батча

def get_batch(split):
    # Получаем случайный батч данных для обучения или валидации
    data_source = train_data if split == 'train' else val_data  # выбираем данные
    # Генерируем случайные начальные позиции
    ix = torch.randint(len(data_source) - block_size, (batch_size,))  # случайные индексы
    # Создаём входную последовательность (x) и целевую (y) - сдвинутую на 1
    x = torch.stack([data_source[i:i+block_size] for i in ix])  # входные последовательности
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])  # целевые (сдвиг на 1)
    x, y = x.to(device), y.to(device)                # перемещаем на GPU/CPU
    return x, y                                       # возвращаем батч

# Тестируем функцию
xb, yb = get_batch('train')                           # получаем тестовый батч
print(f"Форма входа: {xb.shape}")                     # (batch_size, block_size)
print(f"Форма целей: {yb.shape}")                     # (batch_size, block_size)
print(f"Пример входа: {xb[0, :20]}")                  # первые 20 токенов
print(f"Пример цели:  {yb[0, :20]}")                  # первые 20 целевых токенов

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Создаём и обучаем мини-GPT
# ============================================================

# Создаём мини-GPT модель (намного меньше GPT-2)
mini_gpt = GPT(                                       # создаём модель
    vocab_size=vocab_size_mini,                       # размер нашего словаря
    n_embd=128,                                       # размер эмбеддинга (768 у GPT-2)
    n_head=4,                                         # количество голов (12 у GPT-2)
    n_layer=4,                                        # количество слоёв (12 у GPT-2)
    block_size=block_size,                            # длина контекста (1024 у GPT-2)
    dropout=0.1                                       # dropout для регуляризации
).to(device)                                          # перемещаем на GPU/CPU

# Считаем параметры модели
mini_params = sum(p.numel() for p in mini_gpt.parameters())  # общее число параметров
print(f"Мини-GPT параметров: {mini_params:,}")        # выводим (около 500K-1M)

# Настраиваем оптимизатор
optimizer = torch.optim.AdamW(mini_gpt.parameters(), lr=3e-4)  # AdamW оптимизатор

# Обучаем модель
train_losses = []                                     # список для хранения loss на обучении
val_losses = []                                       # список для хранения loss на валидации
eval_interval = 100                                   # интервал оценки
max_steps = 1000                                      # количество шагов обучения

print("Начинаем обучение мини-GPT...")
for step in range(max_steps):                         # цикл обучения
    # Получаем батч и делаем шаг обучения
    xb, yb = get_batch('train')                       # получаем обучающий батч
    logits, loss = mini_gpt(xb, yb)                   # прямой проход
    optimizer.zero_grad(set_to_none=True)             # обнуляем градиенты
    loss.backward()                                   # обратное распространение
    optimizer.step()                                  # обновляем веса

    # Периодически оцениваем на валидации
    if step % eval_interval == 0 or step == max_steps - 1:
        train_losses.append(loss.item())              # сохраняем обучающий loss

        # Оцениваем на валидации
        mini_gpt.eval()                               # режим оценки
        vxb, vyb = get_batch('val')                   # валидационный батч
        with torch.no_grad():                         # без градиентов
            _, val_loss = mini_gpt(vxb, vyb)          # считаем валидационный loss
        val_losses.append(val_loss.item())            # сохраняем валидационный loss
        mini_gpt.train()                              # обратно в режим обучения

        print(f"Шаг {step:4d}/{max_steps} | train loss: {loss.item():.4f} | val loss: {val_loss.item():.4f}")

print("\nОбучение завершено!")

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Визуализация кривых обучения
# ============================================================

# Строим график обучающего и валидационного loss
fig, ax = plt.subplots(figsize=(10, 6))               # создаём фигуру

steps = list(range(0, max_steps + 1, eval_interval))  # шаги оценки
if len(steps) > len(train_losses):                    # если шагов больше чем loss-ов
    steps = steps[:len(train_losses)]                 # обрезаем до размера loss

ax.plot(steps, train_losses, 'b-', label='Train Loss', lw=2, marker='o', markersize=4)  # обучающий loss
ax.plot(steps, val_losses, 'r-', label='Val Loss', lw=2, marker='s', markersize=4)      # валидационный loss
ax.set_xlabel('Шаг обучения', fontsize=12)            # подпись оси X
ax.set_ylabel('Loss (кросс-энтропия)', fontsize=12)   # подпись оси Y
ax.set_title('Обучение мини-GPT: кривые loss', fontsize=14, fontweight='bold')  # заголовок
ax.legend(fontsize=12)                                # легенда
ax.grid(True, alpha=0.3)                              # сетка

# Добавляем аннотацию с минимальным loss
min_val_loss = min(val_losses)                         # минимальный валидационный loss
min_idx = val_losses.index(min_val_loss)              # индекс минимального loss
ax.annotate(f'Лучший val loss: {min_val_loss:.4f}',  # текст аннотации
           xy=(steps[min_idx], min_val_loss),         # позиция на графике
           xytext=(steps[min_idx] + 100, min_val_loss + 0.3),  # позиция текста
           arrowprops=dict(arrowstyle='->', color='red'),  # стрелка
           fontsize=10, color='red')                  # размер и цвет шрифта

plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('mini_gpt_training.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Кривые обучения готовы!")

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Генерация текста обученной мини-GPT
# ============================================================

# Генерируем текст с помощью обученной мини-GPT
mini_gpt.eval()                                       # режим оценки

# Функция для генерации текста нашей моделью
def generate_mini_gpt(start_text, max_new_tokens=200, temperature=0.8, top_k=10):
    # Кодируем начальный текст
    context = torch.tensor([char_to_idx[ch] for ch in start_text], dtype=torch.long)  # кодируем
    context = context.unsqueeze(0).to(device)          # добавляем размерность батча

    # Генерируем новые токены
    output = mini_gpt.generate(context, max_new_tokens, temperature=temperature, top_k=top_k)  # генерируем

    # Декодируем обратно в текст
    generated = ''.join([idx_to_char[idx] for idx in output[0].tolist()])  # декодируем
    return generated                                  # возвращаем результат

# Генерируем с разными начальными промптами
prompts_mini = ["The ", "In a ", "To "]               # начальные промпты
for p in prompts_mini:                                # перебираем промпты
    result = generate_mini_gpt(p, max_new_tokens=150, temperature=0.7)  # генерируем
    print(f"Промпт: '{p}'")
    print(f"Результат: '{result}'")
    print("-" * 60)

---
## Шаг 10. Управление генерацией: промпты и кондиционирование

GPT - это **модель условной генерации**: выход полностью определяется входным промптом. Изменяя промпт, мы направляем генерацию.

### Техники управления генерацией:

1. **Прямое кондиционирование** - задаём стиль/формат в промпте
2. **Few-shot prompting** - показываем примеры в промпте
3. **Контекстное обучение** - модель адаптируется к контексту без дообучения
4. **Форматирование промпта** - структурируем вход для нужного выхода

In [ ]:
# ============================================================
# ШАГ 10: Управление генерацией через промпты
# ============================================================

# Демонстрируем, как разные промпты направляют генерацию по-разному

# 1. Прямое кондиционирование: задаём жанр/стиль
print("=" * 60)
print("1. ПРЯМОЕ КОНДИЦИОНИРОВАНИЕ")
print("=" * 60)

style_prompts = [                                     # промпты в разных стилях
    "Once upon a time in a magical kingdom",           # сказка
    "Breaking news: Scientists have discovered",       # новость
    "In a scientific paper, we demonstrate that",      # научная статья
    "Dear Professor, I am writing to",                 # письмо
]

for prompt in style_prompts:                           # перебираем промпты
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем
    out = model.generate(ids, max_length=80, do_sample=True,  # генерируем
                        temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)  # декодируем
    print(f"\nПромпт: '{prompt}'")
    print(f"Результат: {text[:150]}...")               # первые 150 символов

# 2. Few-shot prompting: показываем примеры
print("\n" + "=" * 60)
print("2. FEW-SHOT PROMPTING (примеры в промпте)")
print("=" * 60)

few_shot_prompt = "Question: What is the capital of France?
" +               "Answer: Paris.

" +               "Question: What is the capital of Germany?
" +               "Answer: Berlin.

" +               "Question: What is the capital of Italy?
" +               "Answer:"                                           # промпт с примерами

ids = tokenizer.encode(few_shot_prompt, return_tensors="pt").to(device)  # токенизируем
out = model.generate(ids, max_length=60, do_sample=False,  # greedy для точности
                    pad_token_id=tokenizer.eos_token_id)
text = tokenizer.decode(out[0], skip_special_tokens=True)  # декодируем
print(f"Промпт с примерами:")
print(text)                                           # выводим результат

---
## Шаг 11. Визуализация сэмплирования: распределения вероятностей на каждом шаге

Посмотрим, как меняется распределение вероятностей по мере генерации текста. На каждом шаге модель предсказывает распределение над всем словарём - визуализируем топ-токены на нескольких шагах.

In [ ]:
# ============================================================
# ШАГ 11: Визуализация распределений на каждом шаге генерации
# ============================================================

# Генерируем текст пошагово и собираем распределения вероятностей
prompt = "The weather today is"                        # промпт
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем

# Сохраняем распределения на каждом шаге
step_distributions = []                                # список распределений
step_chosen_tokens = []                                # выбранные токены на каждом шаге
step_top_tokens = []                                   # топ-токены на каждом шаге

current_ids = input_ids.clone()                        # копируем входные ID

num_steps = 6                                         # количество шагов генерации

for step in range(num_steps):                         # перебираем шаги
    with torch.no_grad():                              # без градиентов
        outputs = model(current_ids)                   # прямой проход
        logits = outputs.logits[0, -1, :].cpu()       # логиты последней позиции

    # Применяем temperature
    temperature = 0.8                                  # temperature для генерации
    scaled_logits = logits / temperature               # масштабируем логиты
    probs = F.softmax(scaled_logits, dim=-1)           # вероятности

    # Получаем топ-15 токенов
    top_k = 15                                         # сколько токенов показать
    top_probs, top_indices = torch.topk(probs, top_k)  # топ-15 вероятностей
    top_token_strs = [tokenizer.decode([idx]) for idx in top_indices.tolist()]  # тексты токенов

    step_distributions.append((top_token_strs, top_probs.numpy()))  # сохраняем распределение

    # Сэмплируем следующий токен
    next_token = torch.multinomial(probs, num_samples=1)  # сэмплируем
    chosen_token_str = tokenizer.decode(next_token)    # текст выбранного токена
    step_chosen_tokens.append(chosen_token_str)        # сохраняем выбор

    # Добавляем к последовательности
    current_ids = torch.cat([current_ids, next_token.unsqueeze(0).to(device)], dim=1)  # добавляем

# Визуализируем распределения на каждом шаге
fig, axes = plt.subplots(2, 3, figsize=(18, 10))      # 2 строки, 3 столбца

for step in range(min(num_steps, 6)):                 # перебираем шаги
    ax = axes[step // 3, step % 3]                    # текущая ось
    token_strs, top_probs_vals = step_distributions[step]  # распределение шага
    chosen = step_chosen_tokens[step]                  # выбранный токен

    # Определяем цвета: выбранный токен - красный, остальные - синие
    colors = ['#E74C3C' if t.strip() == chosen.strip() else '#3498DB' for t in token_strs]

    ax.barh(range(len(token_strs)), top_probs_vals, color=colors)  # столбчатая диаграмма
    ax.set_yticks(range(len(token_strs)))             # деления по Y
    ax.set_yticklabels(token_strs, fontsize=9)        # подписи токенов
    ax.invert_yaxis()                                 # переворачиваем ось
    ax.set_title(f'Шаг {step + 1}: выбрано "{chosen}"\n(красный = выбор)', fontsize=10)
    ax.set_xlabel('Вероятность')

    # Подсвечиваем выбранный токен рамкой
    for j, t in enumerate(token_strs):                # перебираем токены
        if t.strip() == chosen.strip():               # если это выбранный токен
            ax.get_children()[j].set_edgecolor('black')  # чёрная рамка
            ax.get_children()[j].set_linewidth(2)     # толщина рамки

# Показываем финальный текст
final_text = tokenizer.decode(current_ids[0], skip_special_tokens=True)  # декодируем
plt.suptitle(f'Сэмплирование пошагово\nПромпт: "{prompt}" -> Результат: "{final_text}"',
            fontsize=13, fontweight='bold')
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('sampling_visualization.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация сэмплирования готова!")

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Сравнение генерации при разной temperature
# ============================================================

# Генерируем один и тот же промпт при разной temperature и сравниваем
prompt = "In the future, technology will"              # промпт
temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]             # разные значения temperature

results = {}                                          # словарь для результатов

for temp in temperatures:                             # перебираем температуры
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)  # токенизируем
    out = model.generate(ids, max_length=80, do_sample=True,  # генерируем
                        temperature=temp, top_p=0.9,
                        pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)  # декодируем
    results[temp] = text                              # сохраняем результат

# Визуализируем результаты
fig, axes = plt.subplots(len(temperatures), 1, figsize=(14, 3 * len(temperatures)))  # фигура

for idx, temp in enumerate(temperatures):             # перебираем температуры
    ax = axes[idx]                                    # текущая ось
    text = results[temp]                              # текст результата

    # Разбиваем текст на токены для визуализации
    tokens = tokenizer.encode(text)                   # токенизируем
    token_strs = [tokenizer.decode([t]) for t in tokens]  # тексты токенов

    # Показываем текст с цветовой кодировкой
    # Промпт - серый, сгенерированное - цветное
    prompt_len = len(tokenizer.encode(prompt))        # длина промпта в токенах
    x_pos = 0                                         # позиция по X
    for j, tstr in enumerate(token_strs):             # перебираем токены
        width = max(len(tstr) * 0.12 + 0.2, 0.3)     # ширина блока
        if j < prompt_len:                            # если это часть промпта
            color = '#BDC3C7'                         # серый для промпта
        else:                                         # если сгенерированное
            color = plt.cm.plasma(j / len(token_strs))  # цветная шкала
        rect = plt.Rectangle((x_pos, 0.2), width, 0.6,  # прямоугольник
                            facecolor=color, edgecolor='black', lw=0.5)
        ax.add_patch(rect)                            # добавляем прямоугольник
        display_str = tstr.replace('\n', '\\n')    # заменяем переводы строк
        ax.text(x_pos + width/2, 0.5, display_str,   # текст внутри
               ha='center', va='center', fontsize=6)
        x_pos += width + 0.02                         # сдвигаем позицию

    ax.set_xlim(-0.1, x_pos + 0.1)                   # пределы по X
    ax.set_ylim(0, 1)                                 # пределы по Y
    ax.set_title(f'Temperature = {temp}', fontsize=11, fontweight='bold')  # заголовок
    ax.axis('off')                                    # убираем оси

plt.suptitle(f'Влияние Temperature на генерацию\nПромпт: "{prompt}"',
            fontsize=14, fontweight='bold')
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('temperature_comparison.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Сравнение temperature готово!")

---
## Шаг 12. GPT-2 vs GPT-3 vs GPT-4: эволюция языковых моделей

Рассмотрим, как развивалась серия GPT от OpenAI.

### Сравнительная таблица

| Характеристика | GPT-2 | GPT-3 | GPT-4 |
|---------------|-------|-------|-------|
| Год | 2019 | 2020 | 2023 |
| Параметры | 1.5B | 175B | ~1.7T (оценка) |
| Слои | 48 | 96 | ~120 (оценка) |
| Контекст | 1024 | 2048 | 8192-128K |
| Обучающие данные | 40GB | 570GB | ~13T токенов |
| Архитектура | Decoder-only | Decoder-only | Mixture of Experts |
| Multimodal | Нет | Нет | Да (текст + изображения) |

### Ключевые инновации:

- **GPT-2**: Показал, что масштабирование +足够的 данные -> качественная генерация
- **GPT-3**: In-context learning - модель учится из промпта без дообучения
- **GPT-4**: Multimodal + RLHF + значительно лучше рассуждает

In [ ]:
# ============================================================
# ШАГ 12: Визуализация эволюции GPT
# ============================================================

# Данные для визуализации эволюции GPT моделей
models = ['GPT-2\n(2019)', 'GPT-3\n(2020)', 'GPT-3.5\n(2022)', 'GPT-4\n(2023)']
params_billions = [1.5, 175, 175, 1700]               # параметры в миллиардах
context_lengths = [1024, 2048, 4096, 128000]           # максимальная длина контекста
training_data_tb = [0.04, 0.57, 0.57, 13]             # обучающие данные в терабайтах

fig, axes = plt.subplots(1, 3, figsize=(18, 6))       # 3 графика в ряд

# График 1: Количество параметров (логарифмическая шкала)
ax1 = axes[0]                                         # первая ось
colors_params = ['#3498DB', '#E74C3C', '#F39C12', '#2ECC71']  # цвета
bars1 = ax1.bar(models, params_billions, color=colors_params, edgecolor='black', lw=1.5)  # столбцы
ax1.set_yscale('log')                                 # логарифмическая шкала
ax1.set_ylabel('Параметры (миллиарды)', fontsize=11)  # подпись оси Y
ax1.set_title('Количество параметров\n(логарифмическая шкала)', fontsize=12, fontweight='bold')
# Добавляем значения на столбцы
for bar, val in zip(bars1, params_billions):           # перебираем столбцы
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
            f'{val}B', ha='center', va='bottom', fontsize=10, fontweight='bold')

# График 2: Длина контекста
ax2 = axes[1]                                         # вторая ось
bars2 = ax2.bar(models, context_lengths, color=colors_params, edgecolor='black', lw=1.5)  # столбцы
ax2.set_yscale('log')                                 # логарифмическая шкала
ax2.set_ylabel('Макс. длина контекста (токены)', fontsize=11)  # подпись оси Y
ax2.set_title('Длина контекста\n(логарифмическая шкала)', fontsize=12, fontweight='bold')
# Добавляем значения
for bar, val in zip(bars2, context_lengths):           # перебираем столбцы
    label = f'{val//1000}K' if val >= 1000 else str(val)  # формат числа
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

# График 3: Объём обучающих данных
ax3 = axes[2]                                         # третья ось
bars3 = ax3.bar(models, training_data_tb, color=colors_params, edgecolor='black', lw=1.5)  # столбцы
ax3.set_yscale('log')                                 # логарифмическая шкала
ax3.set_ylabel('Обучающие данные (TB)', fontsize=11)  # подпись оси Y
ax3.set_title('Объём обучающих данных\n(логарифмическая шкала)', fontsize=12, fontweight='bold')
# Добавляем значения
for bar, val in zip(bars3, training_data_tb):          # перебираем столбцы
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
            f'{val}TB', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Эволюция моделей GPT: от GPT-2 до GPT-4', fontsize=15, fontweight='bold')
plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('gpt_evolution.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация эволюции GPT готова!")

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Масштабирование - законы масштабирования
# ============================================================

# Визуализируем законы масштабирования (Scaling Laws)
# Идея: больше параметров + больше данных = лучше модель

fig, ax = plt.subplots(figsize=(10, 6))               # создаём фигуру

# Симулируем кривые масштабирования (на основе законов Kaplan et al.)
params_range = np.logspace(7, 12, 100)                # от 10M до 1T параметров

# Loss уменьшается как степенной закон: L(N) ~ N^{-alpha}
alpha = 0.076                                         # коэффициент из статьи
loss_min = 1.5                                        # минимальный achievable loss
loss_curve = loss_min + 3.0 * (params_range / 1e7) ** (-alpha)  # кривая loss от параметров

ax.plot(params_range, loss_curve, 'b-', lw=3, label='Закон масштабирования')  # кривая

# Отмечаем конкретные модели
model_params = {'GPT-2 Small': 124e6, 'GPT-2 XL': 1.5e9,  # параметры моделей
                'GPT-3': 175e9, 'GPT-4 (оценка)': 1.7e12}
model_colors = ['#3498DB', '#9B59B6', '#E74C3C', '#2ECC71']  # цвета

for (name, params), color in zip(model_params.items(), model_colors):  # перебираем модели
    loss_val = loss_min + 3.0 * (params / 1e7) ** (-alpha)  # loss модели
    ax.scatter([params], [loss_val], s=150, color=color, zorder=5, edgecolors='black')  # точка
    ax.annotate(name, xy=(params, loss_val),          # аннотация
               xytext=(10, 10), textcoords='offset points',
               fontsize=9, fontweight='bold', color=color)

ax.set_xscale('log')                                 # логарифмическая ось X
ax.set_xlabel('Количество параметров', fontsize=12)   # подпись оси X
ax.set_ylabel('Loss (кросс-энтропия)', fontsize=12)   # подпись оси Y
ax.set_title('Законы масштабирования\nБольше параметров = ниже loss = лучше модель', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)                                # легенда
ax.grid(True, alpha=0.3)                              # сетка

plt.tight_layout()                                    # автоматическая подстройка
plt.savefig('scaling_laws.png', dpi=150, bbox_inches='tight')  # сохраняем
plt.show()                                            # отображаем
print("Визуализация законов масштабирования готова!")

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

---

### Задание 1. Исследование стратегий генерации

Сгенерируйте один и тот же промпт всеми 4 стратегиями (greedy, beam, top-k, top-p).
Изменяйте параметры и наблюдайте за разницей. Напишите выводы:
- Какая стратегия даёт самый разнообразный текст?
- Какая - самый предсказуемый?
- При каких параметрах текст начинает "галлюцинировать"?

---

### Задание 2. Температурный эксперимент

Для промпта "The meaning of life is" сгенерируйте текст при temperature = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0].
Для каждого значения запишите:
- Насколько осмысленным получился текст?
- Сколько уникальных слов?
- Есть ли повторения?

---

### Задание 3. Обучение мини-GPT на другом тексте

Замените обучающий текст на любой другой (например, стихи Пушкина, код на Python, рецепты).
Обучите мини-GPT и проанализируйте:
- Сколько шагов нужно для сходимости?
- Насколько осмысленный текст генерируется?
- Как влияет размер словаря на скорость обучения?

---

### Задание 4. Анализ каузальной маски

Измените модель GPT так, чтобы убрать каузальную маску (позвольте токенам видеть будущее).
Сгенерируйте текст и сравните с оригинальной моделью. Объясните:
- Почему текст получается бессмысленным?
- Что именно "ломается" без маски?

---

### Задание 5. Создание промптов для разных задач

Напишите промпты для GPT-2, чтобы она сгенерировала:
1. Стихотворение в стиле определённого поэта
2. Рецепт блюда
3. Программный код
4. Научный абзац
5. Диалог двух персонажей

Для каждой задачи подберите оптимальные параметры генерации (temperature, top-k, top-p).

---

> **Совет:** Не бойтесь ломать код! Именно через ошибки и эксперименты приходит настоящее понимание.

---

---
## Итоги ноутбука

### Что мы изучили:

1. **От BERT к GPT** - энкодер vs декодер, авторегрессионная генерация
2. **Кказуальное внимание** - маска, которая не даёт заглядывать вперёд
3. **Архитектура GPT** - слой за слоем: эмбеддинги, блоки декодера, выход
4. **BPE токенизация** - как GPT разбивает текст на подслова
5. **GPT-2 с HuggingFace** - загрузка и генерация текста
6. **Стратегии генерации** - greedy, beam, top-k, top-p, temperature
7. **Интерактивный генератор** - виджеты для управления генерацией
8. **Мини-GPT с нуля** - от данных до обученной модели
9. **Управление промптами** - как направлять генерацию
10. **Визуализация сэмплирования** - распределения на каждом шаге
11. **Эволюция GPT** - от GPT-2 до GPT-4

### Ключевые выводы:

- GPT генерирует текст **авторегрессионно** - по одному токену за раз
- **Кказуальная маска** - фундаментальное отличие от BERT
- **Temperature, top-k, top-p** - инструменты управления разнообразием
- **Промпт** - главный способ контроля генерации
- **Масштабирование** - больше параметров + данные = лучше модель

> Следующий шаг: попробуйте дообучить GPT-2 на своих данных (Fine-tuning)!

---

## FastAPI-сервер для GPT

Эндпоинты:

- `POST /generate` - генерация текста из промпта
- `POST /chat` - диалоговый режим

Параметры: temperature, top_p, max_length. Доступ через ngrok.

In [ ]:
# Установка зависимостей
!pip install fastapi uvicorn pyngrok transformers torch -q

In [ ]:
# Импорты для GPT FastAPI
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict, Optional
import uvicorn
import threading
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Используем небольшую модель для скорости
MODEL_NAME = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gpt_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Модель {MODEL_NAME} загружена.')

In [ ]:
# FastAPI приложение для GPT
app = FastAPI(title='GPT Text Generation API', version='1.0')

class GenerateRequest(BaseModel):
    prompt: str
    max_length: int = 100
    temperature: float = 0.7
    top_p: float = 0.9

class ChatRequest(BaseModel):
    message: str
    history: List[Dict[str, str]] = []
    max_length: int = 100
    temperature: float = 0.7

@app.post('/generate')
def generate_text(req: GenerateRequest):
    # Генерация текста из промпта
    inputs = tokenizer.encode(req.prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = gpt_model.generate(
            inputs,
            max_length=inputs.shape[1] + req.max_length,
            temperature=max(req.temperature, 0.01),
            top_p=req.top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = text[len(req.prompt):]
    return {
        'prompt': req.prompt,
        'generated': generated,
        'full': text,
        'params': {'max_length': req.max_length, 'temperature': req.temperature, 'top_p': req.top_p}
    }

@app.post('/chat')
def chat(req: ChatRequest):
    # Простой диалоговый режим: конкатенируем историю
    context = ''
    for msg in req.history:
        context += f"{msg.get('role', 'user')}: {msg.get('content', '')}\n"
    context += f"user: {req.message}\nassistant:"
    inputs = tokenizer.encode(context, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = gpt_model.generate(
            inputs,
            max_length=inputs.shape[1] + req.max_length,
            temperature=max(req.temperature, 0.01),
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reply = text[len(context):].strip().split('\n')[0]
    return {
        'message': req.message,
        'reply': reply,
        'context': context
    }

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_NAME}

print('FastAPI приложение GPT готово.')

In [ ]:
# Запуск через ngrok
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f'API: {public_url}')
print(f'Docs: {public_url}/docs')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
import time; time.sleep(3)
print('Сервер запущен.')

In [ ]:
# Тестирование GPT API
import requests
base = str(public_url)

# 1. Generate
r = requests.post(f'{base}/generate', json={
    'prompt': 'Once upon a time',
    'max_length': 50,
    'temperature': 0.7,
    'top_p': 0.9
})
print('Generate:', r.json()['generated'])

# 2. Chat
r = requests.post(f'{base}/chat', json={
    'message': 'Hello, who are you?',
    'history': [],
    'max_length': 50
})
print('\nChat reply:', r.json()['reply'])

## Интерактивные виджеты для GPT

Управляйте генерацией через слайдеры и текстовое поле.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=2.0, step=0.1, description='Temperature:')
top_p_slider = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description='top_p:')
max_len_slider = widgets.IntSlider(value=100, min=20, max=500, step=20, description='max_length:')
prompt_input = widgets.Textarea(value='The future of AI is', description='Промпт:')
gen_btn = widgets.Button(description='Сгенерировать', button_style='primary', icon='play')
out = widgets.Output()

def on_gen(b):
    with out:
        clear_output()
        print(f'temperature={temp_slider.value}, top_p={top_p_slider.value}, max_length={max_len_slider.value}')
        print(f'Промпт: {prompt_input.value}\n')
        try:
            r = requests.post(f'{str(public_url)}/generate', json={
                'prompt': prompt_input.value,
                'max_length': max_len_slider.value,
                'temperature': temp_slider.value,
                'top_p': top_p_slider.value
            })
            print('Сгенерировано:')
            print(r.json()['generated'])
        except Exception as e:
            print(f'Ошибка: {e}. Запустите сервер выше.')

gen_btn.on_click(on_gen)
display(temp_slider, top_p_slider, max_len_slider, prompt_input, gen_btn, out)